# 02 · LoRA from scratch
A frozen linear layer plus a low-rank update B·A. The adapter starts as a no-op and merges back exactly.

In [1]:
import sys, json
sys.path.insert(0, '..')
import torch
from gaf.common import device
dev = device(); print(torch.__version__, dev)

2.15.0 mps


In [2]:
from gaf.lora import LoRALinear
torch.manual_seed(0)
base = torch.nn.Linear(128, 128)
l = LoRALinear(base, r=8, alpha=16)
x = torch.randn(4, 128)
print('identity at init:', torch.allclose(l(x), base(x)))
with torch.no_grad(): l.B.normal_(std=0.02)
print('merged == adapter:', torch.allclose(l(x), l.merged()(x), atol=1e-5))
print('trainable', l.A.numel() + l.B.numel(), 'vs full', base.weight.numel() + base.bias.numel())

identity at init: True
merged == adapter: True
trainable 2048 vs full 16512


In [3]:
r = json.load(open('../results/lora.json'))
for run in r['runs']:
    print(f"{run['method']:18} trainable {run['trainable']:>8,}  val(new) {run['val_new']:.3f}  val(original) {run['val_original']:.3f}  {run['seconds']}s")

no fine-tuning     trainable        0  val(new) 2.193  val(original) 1.516  0.0s
full fine-tuning   trainable  824,064  val(new) 1.725  val(original) 1.747  14.9s
LoRA r=2           trainable   18,432  val(new) 1.754  val(original) 1.718  19.7s
LoRA r=8           trainable   73,728  val(new) 1.758  val(original) 1.642  20.9s
LoRA r=32          trainable  294,912  val(new) 1.738  val(original) 1.825  19.9s
